# Análise ODBC — `dados_odbc_01-04.xlsx`

Extração **jan–abr/2026** (conforme nome do arquivo). Colunas alinhadas à visão da [[Analise Base Financeira]] (`codcen` / `descen`, `codcdc` / `descdc`, valores, filial); neste export o `lancamento` já vem como data e os valores como número.

Execute a célula abaixo para o resumo exploratório.

In [36]:
# Análise inicial — dados_odbc_01-04.xlsx (ODBC jan–abr/2026)
# Contexto de negócio: ver 00-Zettlelkasten/Analise Base Financeira.md
# (CC 1.x compras / 2.x receitas; descdc COMPRAS/VENDAS DE SUCATAS; filiais G3S vs G&S etc.)

import os
import pandas as pd
from IPython.display import display

_cwd = os.getcwd() #atribui o caminho do current working directory nessa variável "_cwd"
if os.path.isdir(os.path.join(_cwd, "02-Referencias")):
    WORKSPACE = _cwd
elif os.path.isdir(os.path.join(_cwd, "..", "02-Referencias")):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, ".."))
elif os.path.isdir(os.path.join(_cwd, "..", "..", "02-Referencias")):
    WORKSPACE = os.path.normpath(os.path.join(_cwd, "..", ".."))
else:
    raise FileNotFoundError(f"Pasta 02-Referencias não encontrada a partir de: {_cwd}")

PATH_XLSX = os.path.join(WORKSPACE, "02-Referencias", "dados_odbc_01-04.xlsx")

df = pd.read_excel(PATH_XLSX, sheet_name=0)

def _para_num(serie: pd.Series) -> pd.Series:
    if serie.dtype == object or str(serie.dtype) == "string":
        serie = (
            serie.astype(str)
            .str.replace(".", "", regex=False)
            .str.replace(",", ".", regex=False)
        )
    return pd.to_numeric(serie, errors="coerce")

# Corrige export ODBC com colunas deslocadas (valor_bruto = nome do credor)
_vb_ok = _para_num(df["valor_bruto"]).notna().mean() if "valor_bruto" in df.columns else 1.0
if _vb_ok < 0.5:
  _cols = set(df.columns)
  if "Unnamed: 16" in _cols and "Unnamed: 17" in _cols:
    df["nome"] = df["valor_bruto"].astype(str)
    df["valor_bruto"] = _para_num(df["Unnamed: 16"])
    df["filial"] = df["Unnamed: 17"].astype(str)
    if "Unnamed: 14" in _cols:
      df["documento"] = df["Unnamed: 14"].astype(str)
  elif pd.api.types.is_numeric_dtype(df.get("filial")):
    df["valor_bruto"] = _para_num(df["filial"]).abs()
    if "Unnamed: 17" in _cols:
      df["filial"] = df["Unnamed: 17"].astype(str)

for _col in ("valor_plano", "valor_centro", "valor_bruto", "iterea_valpago"):
  if _col in df.columns:
    df[_col] = _para_num(df[_col])

df["lancamento"] = pd.to_datetime(df["lancamento"])

print("=" * 102)
print("ARQUIVO:", PATH_XLSX)
print("=" * 102)

print(f"Linhas × colunas : {df.shape[0]:,} × {df.shape[1]}")
print(f"Memória (aprox.) : {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

ARQUIVO: c:\Users\julio.santana\Documents\Projects\Cofre_Trabalho\02-Referencias\dados_odbc_01-04.xlsx
Linhas × colunas : 26,969 × 18
Memória (aprox.) : 22.7 MB


In [37]:
print("\n--- Tipos das colunas e qtd de nulos ---")
info = pd.DataFrame({
    "dtype": df.dtypes.astype(str), 
    "nulos": df.isna().sum()
    })
display(info)


--- Tipos das colunas e qtd de nulos ---


,dtype,nulos
codcen,str,0
descen,str,0
codcdc,str,0
descdc,str,0
lancamento,datetime64[us],0
ite_pagrec_vencimento,str,0
iterea_pagamento,str,0
iterea_valpago,float64,0
documento,str,0
codigo_pessoa,int64,0


In [38]:
print("\n--- Valores (`valor_plano` / `valor_centro` / `valor_bruto`) ---")
vp = pd.to_numeric(df["valor_plano"], errors="coerce")
vc = pd.to_numeric(df["valor_centro"], errors="coerce")
print(f"  Linhas com valor_plano ≠ valor_centro : {(vp != vc).sum()}")
vb = pd.to_numeric(df["valor_bruto"], errors="coerce")
print(f"  valor_bruto — min: R$ {vb.min():,.2f} | max: R$ {vb.max():,.2f}")
print(f"  Soma valor_bruto (todas as linhas)     : R$ {vb.sum():,.2f}")

mes = df["lancamento"].dt.to_period("M")
por_mes = df.groupby(mes, observed=True)["valor_bruto"].agg(["sum", "count"]).rename(
    columns={"sum": "soma_valor_bruto", "count": "qtd"}
)
por_mes["soma_valor_bruto"] = por_mes["soma_valor_bruto"].map(lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", "."))
print("\n  Soma `valor_bruto` por mês:")
display(por_mes)




--- Valores (`valor_plano` / `valor_centro` / `valor_bruto`) ---
  Linhas com valor_plano ≠ valor_centro : 0
  valor_bruto — min: R$ 0.01 | max: R$ 1,258,158.92
  Soma valor_bruto (todas as linhas)     : R$ 191,128,913.08

  Soma `valor_bruto` por mês:


,soma_valor_bruto,qtd
lancamento,,
2026-01,"30.352.886,61",5605
2026-02,"33.943.268,36",5565
2026-03,"71.105.625,56",7121
2026-04,"38.265.208,11",6226
2026-05,"17.461.924,44",2452


In [39]:
print("\n--- Período (`lancamento`) ---")
print(f"  Mínimo : {df['lancamento'].min().date()}")
print(f"  Máximo : {df['lancamento'].max().date()}")



mes = df["lancamento"].dt.to_period("M")
print("\n  Lançamentos por mês:")
display(mes.value_counts().sort_index().to_frame("qtd"))


--- Período (`lancamento`) ---
  Mínimo : 2026-01-01
  Máximo : 2026-05-25

  Lançamentos por mês:


,qtd
lancamento,
2026-01,5605
2026-02,5565
2026-03,7121
2026-04,6226
2026-05,2452


In [40]:
# Dimensões a partir de `descen` (mesmo padrão da base financeira: TIPO / DIVISÃO / CIDADE / DEPT)
partes = df["descen"].astype(str).str.split(" / ", n=3, expand=True)
partes.columns = ["tipo_cc", "divisao_cc", "cidade_cc", "dept_cc"]
df_dim = pd.concat([df[["lancamento", "filial", "valor_bruto", "descdc"]], partes], axis=1)

print("\n--- Filiais (`filial`) — top 10 por |valor_bruto| acumulado ---")
fil = df_dim.groupby("filial", observed=True)["valor_bruto"].apply(lambda s: s.abs().sum()).sort_values(ascending=False)
display(fil.head(10).to_frame("soma_abs_valor_bruto"))




--- Filiais (`filial`) — top 10 por |valor_bruto| acumulado ---


,soma_abs_valor_bruto
filial,
RSE,52416150.53
G3S PRUDENTE,36595727.69
G3S CAMPO GRANDE,32203464.38
G3S MARINGA,26506639.79
G3S DOURADOS,13112594.23
G3S LONDRINA,11731207.73
G&S PRUDENTE,5002581.85
G&S BARUERI,4491655.04
G3S CIDADE ALTA,2335240.60


In [41]:
print("\n--- `divisao_cc` (2ª fatia de `descen`) ---")
display(df_dim["divisao_cc"].value_counts().head(25).to_frame("qtd"))



--- `divisao_cc` (2ª fatia de `descen`) ---


,qtd
divisao_cc,
SELETIVA,22298
EKIPA LOCACOES E SERV G&S,1708
PILARES,617
EKIPA LOCACOES RSE,503
EKIPA SERVICOS - G&S,456
EKIPA,375
BRACOFER,338
EKIPA LOCACOES - RSE,263
TRANSMOVE,126


In [42]:

print("\n--- Categorias (`descdc`) — top 12 por volume (soma |valor_bruto|) ---")
cat = df_dim.groupby("descdc", observed=True)["valor_bruto"].apply(lambda s: s.abs().sum()).sort_values(ascending=False)
display(cat.head(12).to_frame("soma_abs_valor_bruto"))



--- Categorias (`descdc`) — top 12 por volume (soma |valor_bruto|) ---


,soma_abs_valor_bruto
descdc,
VENDAS DE SUCATAS,68085283.08
COMPRAS DE SUCATAS,34286399.58
SEGUROS,30180851.07
TRANSPORTE DE SUCATA,6544442.25
JUROS E MULTAS,4686114.45
LOCACAO DE MAQUINAS E EQUIPAMENTOS,4355031.36
RECEITAS DIVERSAS,3851711.34
MANUTENÇÃO DE VEÍCULOS/MAQUINAS,3452692.90
VENDA DE MAQUINAS E EQUIPAMENTOS,3279450.00


In [43]:

print("\n--- Foco sucata (cf. Analise Base Financeira) ---")
m_compra = df_dim["descdc"].eq("COMPRAS DE SUCATAS")
m_venda = df_dim["descdc"].eq("VENDAS DE SUCATAS")
print(f"  COMPRAS DE SUCATAS : {m_compra.sum():,} linhas | R$ {df_dim.loc[m_compra, 'valor_bruto'].sum():,.2f}")
print(f"  VENDAS DE SUCATAS  : {m_venda.sum():,} linhas | R$ {df_dim.loc[m_venda, 'valor_bruto'].sum():,.2f}")

print("\n--- Amostra (5 linhas aleatórias, seed fixo) ---")
display(
    df.sample(5, random_state=42)[
        ["lancamento", "filial", "codcen", "codcdc", "descdc", "valor_bruto", "documento"]
    ]
)


--- Foco sucata (cf. Analise Base Financeira) ---
  COMPRAS DE SUCATAS : 12,015 linhas | R$ 34,286,399.58
  VENDAS DE SUCATAS  : 3,305 linhas | R$ 68,085,283.08

--- Amostra (5 linhas aleatórias, seed fixo) ---


,lancamento,filial,codcen,codcdc,descdc,valor_bruto,documento
7251,2026-04-08,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,1615.0,BOLC-297702
26101,2026-01-08,G3S PRUDENTE,2.2.5.2,5.4.3,PESAGENS AVULSAS,30.0,PA-42761
25821,2026-01-09,G3S CAMPO GRANDE,1.2.7.2,6.1.1,COMPRAS DE SUCATAS,8272.0,BOLC-290145
9020,2026-03-31,G&S DOURADOS,2.7.5.2,5.7.2,FRETE PROPRIO,374.0,CTF42543
8465,2026-04-01,G3S PRUDENTE,2.2.5.2,5.4.3,PESAGENS AVULSAS,60.0,NFE-526


In [44]:
df_seletiva = df[df["codcen"].astype(str).str.startswith(("1.2", "2.2"))]
display(
    df_seletiva.sample(15)[
        ["lancamento", "filial", "codcen", "codcdc", "descdc", "valor_bruto", "documento"]
    ])

,lancamento,filial,codcen,codcdc,descdc,valor_bruto,documento
4899,2026-04-22,G3S CAMPO GRANDE,2.2.7.2,5.4.3,PESAGENS AVULSAS,40.00,NFE-88
11964,2026-03-18,G3S DOURADOS,2.2.2.2,4.1.1,VENDAS DE SUCATAS,4081.00,BOLV-88820
10379,2026-03-26,G3S DOURADOS,2.2.2.1,5.3.3,RENDIMENTO FINANCEIRO,0.17,RENDG3S03260326
12014,2026-03-18,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,27423.00,BOLC-295976
12615,2026-03-16,G3S MARINGA,1.2.4.2,6.1.1,COMPRAS DE SUCATAS,1027.00,BOLC-295695
8493,2026-04-01,G3S CIDADE ALTA,2.2.8.2,4.1.1,VENDAS DE SUCATAS,24.00,BOLV-89148
17729,2026-02-19,G3S CAMPO GRANDE,2.2.7.2,4.1.1,VENDAS DE SUCATAS,40.50,BOLV-88110
14899,2026-03-04,G3S PRUDENTE,2.2.5.2,4.1.1,VENDAS DE SUCATAS,120.60,BOLV-88418
101,2026-05-14,G3S LONDRINA,1.2.3.2,6.1.1,COMPRAS DE SUCATAS,228.00,BOLC-301126
12330,2026-03-17,G3S CIDADE ALTA,1.2.8.2,6.1.1,COMPRAS DE SUCATAS,5760.00,BOLC-295857


In [45]:
# Verifica se há dados faltantes na coluna "valor_bruto"
faltantes = df["valor_bruto"].isnull().sum()
print(f"Número de valores faltantes em 'valor_bruto': {faltantes}")

Número de valores faltantes em 'valor_bruto': 0


In [53]:
# Contas classificadas como custo fixo (plano de contas SAGI)
CONTAS_CUSTO_FIXO = [
    "7.5.13", "7.1.20", "7.5.10", "7.5.1", "7.5.16", "7.5.20", "7.5.3", "7.5.28",
    "7.5.7", "7.6.5", "7.5.18", "7.5.2", "7.5.31", "7.5.36", "7.5.35",
    "7.3.3", "7.3.12", "7.3.6", "7.3.15", "7.3.22", "7.3.2", "7.3.21", "7.3.23", "7.3.1",
    "7.11.7",
]

df_custo_fixo = df_seletiva[df_seletiva["codcdc"].astype(str).isin(CONTAS_CUSTO_FIXO)]

# Identifica os quatro meses presentes no dataframe
meses = df_custo_fixo["lancamento"].dt.to_period("M").unique()[:5]

# Um dataframe por mês, apenas custo fixo
df_mes_1 = df_custo_fixo[df_custo_fixo["lancamento"].dt.to_period("M") == meses[0]]
df_mes_2 = df_custo_fixo[df_custo_fixo["lancamento"].dt.to_period("M") == meses[1]]
df_mes_3 = df_custo_fixo[df_custo_fixo["lancamento"].dt.to_period("M") == meses[2]]
df_mes_4 = df_custo_fixo[df_custo_fixo["lancamento"].dt.to_period("M") == meses[3]]
df_mes_5 = df_custo_fixo[df_custo_fixo["lancamento"].dt.to_period("M") == meses[3]]

In [54]:
# Calcula a soma do valor bruto por cada conta em "descdc" considerando apenas "df_seletiva"
print("Janeiro")
soma_bruto_por_conta = df_mes_1.groupby("descdc", observed=True)["valor_bruto"].sum().sort_values(ascending=False)
display(soma_bruto_por_conta.to_frame("soma_valor_bruto"))

print("Fevereiro")
soma_bruto_por_conta = df_mes_2.groupby("descdc", observed=True)["valor_bruto"].sum().sort_values(ascending=False)
display(soma_bruto_por_conta.to_frame("soma_valor_bruto"))

print("Março")
soma_bruto_por_conta = df_mes_3.groupby("descdc", observed=True)["valor_bruto"].sum().sort_values(ascending=False)
display(soma_bruto_por_conta.to_frame("soma_valor_bruto"))

print("Abril")
soma_bruto_por_conta = df_mes_4.groupby("descdc", observed=True)["valor_bruto"].sum().sort_values(ascending=False)
display(soma_bruto_por_conta.to_frame("soma_valor_bruto"))

print("Maio")
soma_bruto_por_conta = df_mes_5.groupby("descdc", observed=True)["valor_bruto"].sum().sort_values(ascending=False)
display(soma_bruto_por_conta.to_frame("soma_valor_bruto"))

Janeiro


,soma_valor_bruto
descdc,
EMPRESTIMOS,115953.90
SISTEMAS,108856.54
ALUGUEL ADMINISTRATIVO,47812.97
RESCISÕES,12399.85
SALÁRIOS,8430.80
SERVIÇO DE LIMPEZA DO ESCRITÓRIO,4500.00
TELECOMUNICAÇÕES,3862.44
ENERGIA ELÉTRICA,3321.01
BOLSA ESTAGIO,3204.26


Fevereiro


,soma_valor_bruto
descdc,
IPTU PATIO,539144.71
HONORÁRIOS PJ,478210.85
SISTEMAS,120987.59
ALUGUEL ADMINISTRATIVO,110759.97
ENERGIA ELÉTRICA,46526.83
RESCISÕES,22489.51
EMPRESTIMOS,19444.45
SALÁRIOS,15587.69
HONORÁRIOS CONTÁBEIS,14079.00


Março


,soma_valor_bruto
descdc,
HONORÁRIOS PJ,478903.67
EMPRESTIMOS,366254.05
SISTEMAS,152484.91
ALUGUEL ADMINISTRATIVO,96970.97
ENERGIA ELÉTRICA,65562.76
RESCISÕES,52555.51
FGTS,28976.68
INTERNET,9489.78
TELECOMUNICAÇÕES,9002.80


Abril


,soma_valor_bruto
descdc,
HONORÁRIOS PJ,408764.28
SISTEMAS,114086.37
ALUGUEL ADMINISTRATIVO,98966.24
RASTREADOR/ MONITORAMENTO,48083.97
RESCISÕES,31599.08
SALÁRIOS,25528.84
ENERGIA ELÉTRICA,19859.84
SEGUROS,17037.96
FGTS,11934.64


Maio


,soma_valor_bruto
descdc,
HONORÁRIOS PJ,408764.28
SISTEMAS,114086.37
ALUGUEL ADMINISTRATIVO,98966.24
RASTREADOR/ MONITORAMENTO,48083.97
RESCISÕES,31599.08
SALÁRIOS,25528.84
ENERGIA ELÉTRICA,19859.84
SEGUROS,17037.96
FGTS,11934.64


In [56]:
# Calcula a soma do valor bruto por conta para cada mês e junta em uma única tabela
somas_por_mes = pd.DataFrame({
    str(meses[0]): df_mes_1.groupby("descdc", observed=True)["valor_bruto"].sum(),
    str(meses[1]): df_mes_2.groupby("descdc", observed=True)["valor_bruto"].sum(),
    str(meses[2]): df_mes_3.groupby("descdc", observed=True)["valor_bruto"].sum(),
    str(meses[3]): df_mes_4.groupby("descdc", observed=True)["valor_bruto"].sum(),
    str(meses[4]): df_mes_5.groupby("descdc", observed=True)["valor_bruto"].sum(),
}).fillna(0)

# Calcula as diferenças mês a mês
diferencas = somas_por_mes.diff(axis=1).fillna(0)

# Junta tabela de valores brutos e diferenças lado a lado
tabela_com_diferencas = pd.concat([somas_por_mes, diferencas.add_suffix("_dif")], axis=1)

display(tabela_com_diferencas)

# Para facilitar detectar inconsistências, pode-se também destacar as maiores diferenças absolutas
import numpy as np
diferencas_abs = diferencas.abs()
inconsistencias = diferencas_abs[diferencas_abs > (diferencas_abs.mean().mean() + 2 * diferencas_abs.stack().std())]
if not inconsistencias.empty:
    print("Possíveis inconsistências identificadas (diferença mensal muito fora do padrão):")
    display(inconsistencias)

,2026-05,2026-04,2026-03,2026-02,2026-01,2026-05_dif,2026-04_dif,2026-03_dif,2026-02_dif,2026-01_dif
descdc,,,,,,,,,,
ALARME E MONITORAMENTO,2309.11,3570.03,3203.52,2904.89,2904.89,0.0,1260.92,-366.51,-298.63,0.0
ALUGUEL ADMINISTRATIVO,47812.97,110759.97,96970.97,98966.24,98966.24,0.0,62947.00,-13789.00,1995.27,0.0
BOLSA ESTAGIO,3204.26,6989.27,7444.00,7501.34,7501.34,0.0,3785.01,454.73,57.34,0.0
EMPRESTIMOS,115953.90,19444.45,366254.05,0.00,0.00,0.0,-96509.45,346809.60,-366254.05,0.0
ENERGIA ELÉTRICA,3321.01,46526.83,65562.76,19859.84,19859.84,0.0,43205.82,19035.93,-45702.92,0.0
FGTS,910.35,5507.33,28976.68,11934.64,11934.64,0.0,4596.98,23469.35,-17042.04,0.0
HONORÁRIOS CONTÁBEIS,0.00,14079.00,8019.00,11254.00,11254.00,0.0,14079.00,-6060.00,3235.00,0.0
HONORÁRIOS PJ,0.00,478210.85,478903.67,408764.28,408764.28,0.0,478210.85,692.82,-70139.39,0.0
INSS,0.00,502.51,0.00,0.00,0.00,0.0,502.51,-502.51,0.00,0.0


Possíveis inconsistências identificadas (diferença mensal muito fora do padrão):


,2026-05,2026-04,2026-03,2026-02,2026-01
descdc,,,,,
ALARME E MONITORAMENTO,NaN,NaN,NaN,NaN,NaN
ALUGUEL ADMINISTRATIVO,NaN,NaN,NaN,NaN,NaN
BOLSA ESTAGIO,NaN,NaN,NaN,NaN,NaN
EMPRESTIMOS,NaN,NaN,346809.60,366254.05,NaN
ENERGIA ELÉTRICA,NaN,NaN,NaN,NaN,NaN
FGTS,NaN,NaN,NaN,NaN,NaN
HONORÁRIOS CONTÁBEIS,NaN,NaN,NaN,NaN,NaN
HONORÁRIOS PJ,NaN,478210.85,NaN,NaN,NaN
INSS,NaN,NaN,NaN,NaN,NaN
